# Explorar o explotar: el dilema de los bandidos

**Facsímil 10 · Aprendizaje por refuerzo** — capítulo 3 (exploración, *bandits* y validación de
políticas).

Este es el dilema más puro del aprendizaje por interacción, y aparece en cualquier sistema que aprende
mientras actúa: tests A/B de una web, qué anuncio mostrar, qué canción recomendar, qué dosis de un
tratamiento probar. Tienes varias opciones y no sabes cuál es mejor. Cada prueba te da información, pero
también te cuesta (una visita gastada en la versión mala, un cliente que no vuelve). ¿Sigues con la que
parece buena (**explotar**) o arriesgas con otras por si son mejores (**explorar**)? En este cuaderno
mides, en *regret* (lo que pierdes frente a haber jugado perfecto), cuatro estrategias, y ves por qué
explorar **con criterio** bate a explorar al azar, con experimentos sobre el valor de ε, la dificultad
del problema y los mundos que cambian.

### Qué vas a aprender
- El dilema **explorar vs explotar** y por qué los dos extremos son malos.
- Cuatro estrategias: **aleatorio**, **ε-greedy**, **UCB1** y **Thompson sampling**, y la intuición de
  cada una.
- Qué es el **regret** y por qué es la métrica honesta (no «cuánto gané», sino «cuánto dejé de ganar»).
- A *ver* la diferencia en la **curva de regret**, en **qué brazo elige** cada estrategia y en el **% de
  acción óptima** a lo largo del tiempo.
- A medir, con experimentos, el efecto de **ε**, de un **problema difícil**, de la **ε decreciente** y de
  un **mundo no estacionario**.

### Cómo se usa
Las celdas vienen **sin ejecutar**: pulsa ▶ de arriba abajo. Fijamos la semilla del azar para que los
números sean reproducibles.

### Cuánto cuesta
Unos 18 minutos. CPU, sin claves.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(1)

# 6 maquinas con probabilidad de premio desconocida. La 4 es la mejor (0.75).
PROBS = np.array([0.20, 0.45, 0.55, 0.40, 0.75, 0.30])
MEJOR = PROBS.max()
BRAZO_OPTIMO = int(PROBS.argmax())
N = 2000          # tiradas totales
def tira(brazo): return 1.0 if np.random.random() < PROBS[brazo] else 0.0

print(f"{len(PROBS)} maquinas. La mejor es la nº {BRAZO_OPTIMO}, da premio el {MEJOR*100:.0f}% de las veces")
print("(pero el agente NO lo sabe: solo puede aprender tirando y mirando que sale).")
print("\nProbabilidad real de cada maquina:")
for i, p in enumerate(PROBS):
    marca = "  <- la mejor" if i == BRAZO_OPTIMO else ""
    print(f"  maquina {i}: {p:.2f}{marca}")


## 1. El dilema, y por qué los extremos son malos

Imagina que solo **explotas**: tiras siempre de la primera máquina que te dio un premio. Quizá te
anclaste en una mediocre y nunca descubres la buena. Ahora imagina que solo **exploras**: tiras siempre
al azar para «aprender», sin aprovechar nunca lo aprendido. Malgastas. La gracia está en el equilibrio:
explorar **lo justo** para descubrir cuál es la buena, y luego explotarla. Las estrategias que veremos son
formas distintas de lograr ese equilibrio.


## 2. Cuatro formas de decidir

- **Aleatorio:** tira al azar siempre. No aprende nada; es nuestra línea base, el suelo contra el que
  medir a las demás.
- **ε-greedy:** casi siempre tira de la mejor conocida; con probabilidad pequeña ε, tira al azar (explora
  a ciegas). Simple, pero explora para siempre con la misma intensidad.
- **UCB1** (*upper confidence bound*): elige la de mayor «valor estimado **+** bono de incertidumbre».
  Explora lo que ha probado poco, de forma dirigida. Optimismo ante la incertidumbre.
- **Thompson sampling:** mantiene una creencia (una distribución *beta*) por máquina y muestrea de ella;
  explora de forma bayesiana y elegante, proporcional a la probabilidad de que cada una sea la mejor.


In [ ]:
def correr(politica, n=N, epsilon=0.1, semilla=1):
    rng = np.random.default_rng(semilla)
    def tirar(brazo): return 1.0 if rng.random() < PROBS[brazo] else 0.0
    q = np.zeros(len(PROBS)); n_b = np.zeros(len(PROBS)); recompensa = 0.0
    a_beta = np.ones(len(PROBS)); b_beta = np.ones(len(PROBS))   # para Thompson
    curva, optimo = [], []
    for t in range(1, n+1):
        if politica == "aleatorio":
            brazo = rng.integers(len(PROBS))
        elif politica == "epsilon":
            brazo = rng.integers(len(PROBS)) if rng.random() < epsilon else int(np.argmax(q))
        elif politica == "ucb":
            bono = np.sqrt(2*np.log(t) / np.maximum(n_b, 1e-9))
            brazo = int(np.argmax(q + bono)) if t > len(PROBS) else (t-1)
        else:  # thompson
            brazo = int(np.argmax(rng.beta(a_beta, b_beta)))
        r = tirar(brazo)
        n_b[brazo] += 1; q[brazo] += (r - q[brazo]) / n_b[brazo]
        a_beta[brazo] += r; b_beta[brazo] += (1-r); recompensa += r
        curva.append(t*MEJOR - recompensa)              # regret acumulado hasta t
        optimo.append(1.0 if brazo == BRAZO_OPTIMO else 0.0)
    return {"recompensa": recompensa, "regret": np.array(curva),
            "optimo": np.array(optimo), "tiradas": n_b.copy()}

ESTRATEGIAS = ["aleatorio", "epsilon", "ucb", "thompson"]
resultados = {pol: correr(pol) for pol in ESTRATEGIAS}
print(f"{'estrategia':<12}{'premios':>9}{'regret final':>14}{'% optima global':>18}")
for pol in ESTRATEGIAS:
    r = resultados[pol]
    print(f"{pol:<12}{r['recompensa']:>9.0f}{r['regret'][-1]:>14.1f}{100*r['optimo'].mean():>16.1f} %")


## 3. El regret, la métrica honesta

El **regret** es lo que dejas de ganar frente a un oráculo que **siempre** juega la mejor máquina:
cuanto **más bajo, mejor**. No mide «cuánto gané» (que depende de la suerte), sino «cuánto perdí por no
saber desde el principio cuál era la buena». El aleatorio acumula muchísimo (no aprende: tira a ciegas
para siempre). Las otras lo dejan muy atrás porque **descubren** la buena y se quedan con ella. Mira en la
tabla de arriba la columna de regret final y la de % de acción óptima: van de la mano.


## 4. Verlo: la curva de regret

La diferencia se *ve* en cómo crece el regret con el tiempo. Una estrategia que aprende hace que su
curva se **aplane** (deja de acumular regret porque ya explota la buena). La aleatoria crece en línea
recta para siempre. Esa curvatura **es** el aprendizaje.


In [ ]:
plt.figure(figsize=(7, 3.8))
for pol in ESTRATEGIAS:
    estilo = "--" if pol == "aleatorio" else "-"
    plt.plot(resultados[pol]["regret"], estilo, label=pol,
             color="black" if pol == "thompson" else None)
plt.xlabel("tirada"); plt.ylabel("regret acumulado")
plt.title("La curva que aprende se aplana; la aleatoria crece recta")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()
print("Fijate: las estrategias listas frenan (aprenden cual es la buena); la aleatoria, nunca.")


## 5. ¿Qué máquina acabó eligiendo cada una?

El regret cuenta el resultado; este gráfico cuenta el **porqué**. Si dibujamos cuántas veces tiró cada
estrategia de cada máquina, vemos que las listas concentran casi todas sus tiradas en la máquina buena (la
nº 4), mientras que el aleatorio reparte por igual entre las seis, malgastando en las malas.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(11, 2.8), sharey=True)
for ax, pol in zip(axes, ESTRATEGIAS):
    tir = resultados[pol]["tiradas"]
    colores = ["black" if i == BRAZO_OPTIMO else "0.7" for i in range(len(PROBS))]
    ax.bar(range(len(PROBS)), tir, color=colores)
    ax.set_title(pol, fontsize=10); ax.set_xlabel("maquina")
axes[0].set_ylabel("nº de tiradas")
fig.suptitle("Cuanto tira cada estrategia de cada maquina (negro = la mejor)", fontsize=11)
plt.tight_layout(); plt.show()
for pol in ESTRATEGIAS:
    tir = resultados[pol]["tiradas"]
    print(f"{pol:<12} concentro el {100*tir[BRAZO_OPTIMO]/tir.sum():.1f} % de sus tiradas en la mejor maquina")


## 6. El % de acción óptima a lo largo del tiempo

Otra forma de mirar el aprendizaje: ¿qué fracción de las últimas jugadas fue la máquina buena? Suavizamos
con una media móvil. Una estrategia que aprende sube hacia el 100 %; el aleatorio se queda plano en
torno a 1/6 (el azar puro acierta la buena una de cada seis veces).


In [ ]:
def media_movil(x, k=100):
    return np.convolve(x, np.ones(k)/k, mode="valid")

plt.figure(figsize=(7, 3.8))
for pol in ESTRATEGIAS:
    estilo = "--" if pol == "aleatorio" else "-"
    plt.plot(media_movil(resultados[pol]["optimo"]), estilo, label=pol,
             color="black" if pol == "thompson" else None)
plt.axhline(1/len(PROBS), color="0.6", ls=":", lw=1, label="azar puro (1/6)")
plt.xlabel("tirada"); plt.ylabel("% de acciones que son la optima (media movil)")
plt.title("Aprender es subir hacia la maquina buena"); plt.ylim(0, 1.05)
plt.legend(fontsize=8); plt.tight_layout(); plt.show()


## 7. Una sola partida engaña: promediar muchas

Una única simulación depende de la suerte de esa semilla. Para comparar de verdad, repetimos cada
estrategia muchas veces con semillas distintas y promediamos su regret. Así medimos el comportamiento
**típico**, no el de un día con suerte. Es exactamente lo que se hace al validar una política antes de
soltarla en producción.


In [ ]:
REPETICIONES = 40
prom = {}
for pol in ESTRATEGIAS:
    curvas = np.array([correr(pol, semilla=s)["regret"] for s in range(REPETICIONES)])
    prom[pol] = (curvas.mean(0), curvas.std(0))

plt.figure(figsize=(7, 3.8))
for pol in ESTRATEGIAS:
    media, desv = prom[pol]
    estilo = "--" if pol == "aleatorio" else "-"
    col = "black" if pol == "thompson" else None
    linea, = plt.plot(media, estilo, label=pol, color=col)
    plt.fill_between(range(len(media)), media-desv, media+desv, alpha=0.15,
                     color=linea.get_color())
plt.xlabel("tirada"); plt.ylabel("regret acumulado (media de 40 partidas)")
plt.title("Regret promedio; la banda es la variabilidad entre partidas")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()
print(f"{'estrategia':<12}{'regret final medio':>20}{'desviacion':>14}")
for pol in ESTRATEGIAS:
    media, desv = prom[pol]
    print(f"{pol:<12}{media[-1]:>20.1f}{desv[-1]:>14.1f}")


## 8. Experimento: ¿cuánta exploración conviene en ε-greedy?

ε es la perilla de la exploración: la fracción de tiradas que ε-greedy «tira a ciegas». Con ε muy bajo
explora poco y tarda en encontrar la buena; con ε muy alto malgasta tiradas en máquinas que ya sabe que
son malas. Hay un punto dulce. Lo medimos promediando varias partidas por cada valor de ε.


In [ ]:
print(f"{'epsilon':>8}{'regret final medio (40 partidas)':>35}")
for eps in [0.01, 0.05, 0.1, 0.3, 0.5]:
    finales = [correr("epsilon", epsilon=eps, semilla=s)["regret"][-1] for s in range(40)]
    print(f"{eps:>8}{np.mean(finales):>30.1f}")
print("\nNi muy poco (tarda en hallar la buena) ni mucho (malgasta tiradas en las malas):")
print("hay un valor intermedio de epsilon que minimiza el regret.")


## 9. Experimento: un problema fácil y otro difícil

Cuando las máquinas se parecen mucho, distinguir la buena cuesta un mundo. Lo comprobamos con dos mundos
nuevos: uno **fácil** (la mejor destaca con claridad) y otro **difícil** (todas casi iguales). Aquí hay un
matiz precioso: en el difícil **fallas mucho más** (te cuesta separar lo bueno de lo casi-bueno), así que
tu **% de acción óptima cae en picado**. Pero el **regret en puntos** no tiene por qué subir: como las
máquinas casi empatan, cada error vale poquísimo. Por eso miramos las dos cosas a la vez.


In [ ]:
def evalua_probs(probs, politica, semilla=0, n=N):
    rng = np.random.default_rng(semilla)
    mejor = probs.max(); optimo = int(probs.argmax())
    q = np.zeros(len(probs)); n_b = np.zeros(len(probs)); rec = 0.0; aciertos = 0
    a_b = np.ones(len(probs)); b_b = np.ones(len(probs))
    for t in range(1, n+1):
        if politica == "epsilon":
            brazo = rng.integers(len(probs)) if rng.random() < 0.1 else int(np.argmax(q))
        elif politica == "ucb":
            bono = np.sqrt(2*np.log(t)/np.maximum(n_b, 1e-9))
            brazo = int(np.argmax(q+bono)) if t > len(probs) else (t-1)
        else:
            brazo = int(np.argmax(rng.beta(a_b, b_b)))
        r = 1.0 if rng.random() < probs[brazo] else 0.0
        n_b[brazo] += 1; q[brazo] += (r-q[brazo])/n_b[brazo]
        a_b[brazo] += r; b_b[brazo] += (1-r); rec += r
        aciertos += (brazo == optimo)
    return t*mejor - rec, 100*aciertos/n

FACIL = np.array([0.10, 0.15, 0.20, 0.25, 0.80, 0.20])
DIFICIL = np.array([0.50, 0.51, 0.52, 0.53, 0.55, 0.52])
print(f"{'estrategia':<12}{'% optima facil':>16}{'% optima dificil':>18}")
for pol in ["epsilon", "ucb", "thompson"]:
    of = np.mean([evalua_probs(FACIL, pol, s)[1] for s in range(40)])
    od = np.mean([evalua_probs(DIFICIL, pol, s)[1] for s in range(40)])
    print(f"{pol:<12}{of:>15.1f}%{od:>17.1f}%")
print("\nEn el mundo dificil el % de acciones optimas se desploma: no logras distinguir la buena.")
print("Esa es la cara honesta de la dificultad; el regret en puntos enganaria, porque cada error")
print("cuesta una miseria cuando todas las maquinas casi empatan.")


## 10. Experimento: ε que decrece con el tiempo

Una idea clásica: explorar mucho al principio (cuando no sabes nada) y cada vez menos según vas
aprendiendo. Es la ε **decreciente** (ε = 1/t suavizado). Combina lo mejor de los dos mundos: descubre
rápido la buena y luego deja de malgastar. La comparamos con la ε fija de 0,1.


In [ ]:
def correr_eps_decreciente(semilla=0, n=N):
    rng = np.random.default_rng(semilla)
    q = np.zeros(len(PROBS)); n_b = np.zeros(len(PROBS)); rec = 0.0; curva = []
    for t in range(1, n+1):
        eps = min(1.0, 5.0/t)        # mucha exploracion al principio, casi nada al final
        brazo = rng.integers(len(PROBS)) if rng.random() < eps else int(np.argmax(q))
        r = 1.0 if rng.random() < PROBS[brazo] else 0.0
        n_b[brazo] += 1; q[brazo] += (r-q[brazo])/n_b[brazo]; rec += r
        curva.append(t*MEJOR - rec)
    return np.array(curva)

fija = np.array([correr("epsilon", epsilon=0.1, semilla=s)["regret"] for s in range(40)]).mean(0)
decr = np.array([correr_eps_decreciente(s) for s in range(40)]).mean(0)
plt.figure(figsize=(7, 3.6))
plt.plot(fija, label="epsilon fijo (0.1)")
plt.plot(decr, color="black", label="epsilon decreciente (5/t)")
plt.xlabel("tirada"); plt.ylabel("regret acumulado (media 40)")
plt.title("Explorar mucho al principio y menos despues"); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"regret final - epsilon fijo: {fija[-1]:.1f}  |  epsilon decreciente: {decr[-1]:.1f}")
print("La decreciente frena antes su regret: deja de malgastar cuando ya sabe cual es la buena.")


## 11. Experimento: el mundo cambia (bandits no estacionarios)

Hasta ahora la mejor máquina era siempre la misma. En la realidad las modas cambian: la versión de la web
que más gustaba en enero puede dejar de funcionar en marzo. Hacemos que, a mitad de partida, **otra**
máquina pase a ser la mejor. La media de toda la historia (lo que veníamos usando) reacciona con pereza;
una media que **olvida** lo viejo (paso constante α) se adapta mucho antes. Para aprender en un mundo que
cambia hay que **seguir explorando** y **dar más peso a lo reciente**.


In [ ]:
def correr_no_estacionario(usa_olvido, semilla=0, n=4000, alpha=0.1, eps=0.1):
    rng = np.random.default_rng(semilla)
    probs = np.array([0.20, 0.45, 0.55, 0.40, 0.75, 0.30])
    q = np.zeros(len(probs)); n_b = np.zeros(len(probs)); aciertos = []
    for t in range(1, n+1):
        if t == n//2:                       # giro brusco: la 0 pasa a ser la mejor
            probs = np.array([0.85, 0.45, 0.30, 0.40, 0.30, 0.30])
        mejor_actual = int(probs.argmax())
        brazo = rng.integers(len(probs)) if rng.random() < eps else int(np.argmax(q))
        r = 1.0 if rng.random() < probs[brazo] else 0.0
        n_b[brazo] += 1
        paso = alpha if usa_olvido else 1.0/n_b[brazo]   # olvido vs media de toda la historia
        q[brazo] += paso*(r - q[brazo])
        aciertos.append(1.0 if brazo == mejor_actual else 0.0)
    return np.array(aciertos)

mm = lambda x, k=200: np.convolve(x, np.ones(k)/k, mode="valid")
sin_olvido = np.array([correr_no_estacionario(False, s) for s in range(30)]).mean(0)
con_olvido = np.array([correr_no_estacionario(True, s) for s in range(30)]).mean(0)
plt.figure(figsize=(7, 3.6))
plt.plot(mm(sin_olvido), label="media de toda la historia")
plt.plot(mm(con_olvido), color="black", label="media que olvida (paso fijo)")
plt.axvline(4000//2 - 100, color="0.6", ls=":", label="el mundo cambia")
plt.xlabel("tirada"); plt.ylabel("% accion optima (media movil)")
plt.title("Cuando el mundo cambia, conviene olvidar lo viejo"); plt.legend(fontsize=8)
plt.ylim(0, 1.05); plt.tight_layout(); plt.show()
print("Tras el giro, la media que olvida recupera la accion optima mucho antes que la que recuerda todo.")


## 12. Experimento: optimismo inicial, explorar sin ε

Hay una forma astuta de explorar **sin** tirar nunca al azar: empezar **optimista**. Si le dices al agente
que, de entrada, **todas** las máquinas dan premio siempre (valor inicial 1,0) y luego es puramente
avaricioso, cada máquina que pruebe lo «decepcionará» (su valor estimado bajará), y eso le empuja a probar
las demás, una por una, hasta que la realidad se asienta. Lo comparamos con un avaricioso pesimista (valor
inicial 0, que se ancla en la primera que paga) y con ε-greedy.


In [ ]:
def correr_optimista(q0, semilla=0, n=N):
    rng = np.random.default_rng(semilla)
    q = np.full(len(PROBS), float(q0)); n_b = np.zeros(len(PROBS)); rec = 0.0; curva = []
    for t in range(1, n+1):
        brazo = int(np.argmax(q))                       # avaricioso puro, sin azar
        r = 1.0 if rng.random() < PROBS[brazo] else 0.0
        n_b[brazo] += 1; q[brazo] += (r - q[brazo])/n_b[brazo]; rec += r
        curva.append(t*MEJOR - rec)
    return np.array(curva)

opt = np.array([correr_optimista(1.0, s) for s in range(40)]).mean(0)
pes = np.array([correr_optimista(0.0, s) for s in range(40)]).mean(0)
eps = np.array([correr("epsilon", epsilon=0.1, semilla=s)["regret"] for s in range(40)]).mean(0)
plt.figure(figsize=(7, 3.6))
plt.plot(pes, label="avaricioso pesimista (q0=0)")
plt.plot(eps, label="epsilon-greedy (0.1)")
plt.plot(opt, color="black", label="avaricioso optimista (q0=1)")
plt.xlabel("tirada"); plt.ylabel("regret acumulado (media 40)")
plt.title("El optimismo inicial explora sin necesidad de azar"); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"regret final - pesimista (q0=0): {pes[-1]:.1f}")
print(f"regret final - epsilon-greedy:   {eps[-1]:.1f}")
print(f"regret final - optimista (q0=1): {opt[-1]:.1f}")
print("\nEl pesimista se ancla en la primera maquina que paga y acumula muchisimo regret. El optimista,")
print("solo con ser avaricioso desde valores altos, explora al principio y lo deja muy atras; aqui no")
print("llega al nivel de epsilon-greedy, pero explora SIN una sola tirada al azar, que es lo elegante.")


## 13. Pruébalo tú

1. **Sube ε a 0,5** en la celda del paso 2 (`correr("epsilon", epsilon=0.5)`): explora la mitad del
   tiempo. Compara con la tabla del experimento 8: demasiada exploración también se paga.
2. **Acerca las máquinas** usando el mundo `DIFICIL`: cuando las opciones se parecen, el % de acción
   óptima se desploma y todas las estrategias sufren.
3. **Alarga el horizonte** a `N = 20000`: ¿se nota más la ventaja de UCB y Thompson a largo plazo? Sus
   garantías teóricas son asintóticas.
4. **Cambia el giro** del paso 11 para que la mejor pase a ser la **peor**: ¿cuánto tarda en recuperarse
   la media que olvida frente a la que recuerda todo?
5. **Sube el optimismo inicial** a `q0=5` en el paso 12: con más optimismo explora más al principio.
   ¿Hay un punto en que demasiado optimismo también estorba?


## 14. Errores comunes

- **Explotar demasiado pronto.** Si te quedas con la primera que parece buena, puedes anclarte en una
  mediocre. Hay que explorar lo justo primero (lo viste en el experimento de ε).
- **Explorar para siempre con la misma intensidad** (ε-greedy con ε fijo y alto): malgastas tiradas en
  máquinas que ya sabes que son malas. La ε decreciente lo arregla.
- **Mirar «cuánto gané» en vez del regret.** Lo que ganaste depende de la suerte; el regret mide tu
  decisión frente a lo óptimo, y promediar muchas partidas (paso 7) quita el ruido de una semilla con
  suerte.
- **Asumir que el mundo no cambia.** Muchos problemas reales son no estacionarios: la media de toda la
  historia reacciona tarde; conviene dar más peso a lo reciente y seguir explorando.
- **Comparar estrategias con una sola partida.** Una semilla afortunada puede mentir; la media con su
  banda de variabilidad es lo que de verdad decide.


## 15. Qué te llevas

- El dilema **explorar/explotar** aparece siempre que aprendes mientras actúas: cada decisión es a la vez
  una apuesta y un experimento.
- Estrategias como **UCB** y **Thompson** baten de calle al azar porque exploran **de forma dirigida**
  (prueban lo incierto, no lo aleatorio); lo confirmaste en el regret, en qué brazo eligen y en el % de
  acción óptima.
- El **regret** es la métrica honesta: no «cuánto gané», sino «cuánto dejé de ganar frente a lo óptimo».
  Su curva, al aplanarse, es la imagen del aprendizaje, y solo se mide bien **promediando** muchas partidas.
- La **dificultad del problema**, el valor de **ε**, una **ε decreciente** y un **mundo que cambia** mueven
  el resultado tanto como la estrategia: medir cada caso es parte del oficio.

**Para seguir:** los *bandits* son el RL más simple (un solo paso). El facsímil escala esto a secuencias
de decisiones (MDP, el cuaderno anterior) y a aprender de preferencias humanas (RLHF, capítulo 5).


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*